---
title: "01. Overview & the golden path"
description: "The four planes, the fixed invariants, the golden path a model travels, and the two-part build: features crystallize on Docker Compose first, then transfer to Azure behind an explicit contract."
---

## Outcome

By the end of this chapter you can describe the whole platform on one page: the
four planes it is built from, the invariants that never change, the *golden path*
a model travels from data to production, and the phased order in which we build
it. Every later chapter fills in one phase of this map.

The build runs in two parts. **Part I** stands the entire platform up locally on
Docker Compose and builds every feature on top of it; **Part II** ports the result
to Azure behind an explicit environment contract (chapter 08). Azure terms in this
chapter name that second part unless stated otherwise.

The guiding rule for this course is **the fewest moving parts that deliver
reproducibility, honest evaluation, scheduled/on-demand workflows, and observable
operations** — and no more. We deliberately reject an enterprise control-plane
build (durable orchestration engine, hash-chained release ledgers, a bespoke
broker) because it costs more to build and operate than a small ML team can
sustain and buys guarantees we do not yet need.


## The four planes

The platform is four planes plus a thin dashboard. Everything else follows. Both
parts of the course build the same four planes; only the concrete infrastructure
underneath them differs.

| Plane | Responsibility | Part I (Compose) | Part II (Azure) |
|---|---|---|---|
| **Execution** | Run every workflow as an ephemeral, image-pinned task | One-shot Compose services + the local runner | Azure Container Apps **Jobs** |
| **Model lifecycle** | Track experiments, register versions, store artifacts | **Self-hosted MLflow** container + Postgres + MinIO | Self-hosted MLflow ACA App + Postgres + Blob Storage |
| **Operational state** | Record status/output/error for every run, with batch granularity | Postgres `results` DB | Postgres `results` DB (unchanged) |
| **Serving** | Optional online HTTP inference at an exact model version | FastAPI container (`serving`) | Serving ACA App |

```mermaid
flowchart TD
    DASH["Dashboard<br/>catalog + launcher + links"]
    JOBS["Jobs (train / eval / batch / task)<br/>ephemeral, image-pinned"]
    MLF["Self-hosted MLflow<br/>registry + tracking"]
    RDB["Results DB<br/>run state"]
    BLOB["Object store (artifacts)"]
    LOGS["Container logs<br/>operational signals"]

    DASH -->|reads status| RDB
    DASH -->|starts| JOBS
    DASH -->|deep-links| MLF
    JOBS -->|read/write model versions| MLF
    JOBS -->|write runs| RDB
    JOBS -->|large payloads| BLOB
    JOBS -->|stdout/stderr| LOGS
    MLF --> BLOB
```

Part I runs every box in this diagram on Compose; Part II swaps the infrastructure
underneath without changing the arrows. The **dashboard** is a read-and-launch
surface over these planes; it holds no authoritative state of its own.


## Fixed invariants

These hold across every chapter of both parts. Changing any of them is an
architecture decision, not an implementation detail.

1. **Ephemeral, image-pinned jobs are the execution plane** — Compose one-shots
   plus the local runner in Part I, Azure Container Apps Jobs in Part II. Every
   workflow (train, eval, batch, ad-hoc) runs from a pinned image digest and
   scales to zero when idle; a code deploy is always just an image-digest bump,
   so it can never strand a stale worker on old code.
2. **No workflow control-plane service** — linear multi-step workflows are one
   script in one job run. No orchestration engine, no durable-execution framework.
3. **No application broker by default** — no message bus, Redis broker, or Celery
   fleet in the baseline. Fan-out is parent/child rows in the results DB.
4. **Self-hosted MLflow is the tracking + model registry**, pinned to an exact
   version — the registered **version number is the canonical model identity**.
5. **A generic results DB is the run store** — one table records status/output/
   error for every job, with parent/child rows for batch granularity.
6. **MLflow is scoped to model lifecycle** — batch inference only *reads* a pinned
   version; its state lives in the results DB, not as MLflow runs.
7. **GitHub Actions is CI/CD only** — build/test/scan images and update workload
   definitions in both environments. Never a scheduler or orchestrator.
8. **Least-privilege identities; OIDC for CI on Azure** — each workload has its
   own identity with minimal roles: managed identities in Part II, explicit demo
   credentials in Part I. No shared broad identity, no secrets in images.
9. **Distributed/multi-GPU training is an admission-gated exception** — Azure ML
   `command` jobs against min-zero clusters (a Part II path), logging to the same
   MLflow.


## The golden path

A single path a model travels from data to production. Every step runs as a
short-lived job in the execution plane, every model is a registry version, every
run is a results-DB record, and every human action is audited.

```mermaid
flowchart TD
    DATA["tracked dataset"]
    TRAIN["train job"]
    VER["MLflow run + registered version"]
    EVAL["eval job<br/>metrics + results-DB record"]
    PROMOTE["promotion gate<br/>alias + exact-version repin"]
    BATCH["batch job (scheduled/manual)<br/>parent/child result rows"]
    SERVE["serving app (optional)<br/>exact model version"]
    OPS["results dashboard + logs<br/>two batch alerts on Azure"]

    DATA --> TRAIN --> VER --> EVAL --> PROMOTE
    PROMOTE --> BATCH
    PROMOTE --> SERVE
    BATCH --> OPS
    SERVE --> OPS
```

Part I runs every step of this path on Compose; Part II swaps the infrastructure
underneath without changing the arrows. Promotion is deliberately blocked until
the exact candidate version has a passing evaluation record.


## The phased build, in two parts

Features crystallize on Compose first: Part I builds the platform locally and
exercises every feature against that one stack. [Chapter 08](08-environment-contract.ipynb)
then writes down the environment contract (variables, schemas, trigger APIs,
promotion rules, and behavioral assertions) that makes the port to Azure
mechanical. Every Part II chapter maps its deployment adapter onto what Part I
already proved.

**Part I: Local platform (Compose)**

| Ch | What ships | Why it matters |
|---|---|---|
| **02** Local platform foundation | the whole footprint on one laptop: nine containers, one Compose file | Everything has a home |
| **03** Reproducible training & registry | train/eval jobs logging to MLflow + results-DB records | Reproducible models with an enforceable quality gate |
| **04** Results DB & batch workflows | results-DB module + batch workflows (parent/child + bounded retries); scheduled execution arrives in Part II | The operational backbone exists and is exercised |
| **05** Online serving & promotion | serving container + evaluation-gated, version-based promotion/rollback | Models reach consumers with safe rollback |
| **06** Observability & dashboard | results visibility, container logs, and a catalog/launcher dashboard | The team can see and launch everything |
| **07** LLM release artifacts | pyfunc packaging + evaluator reusing the same paths | An LLM ships through existing machinery |
| **08** The environment contract | the transfer spec: env vars, schemas, trigger APIs, promotion rules, separate adapters with shared behavioral assertions | The Azure port preserves shared behavior while changing adapters |

**Part II: Production (Azure)**

| Ch | What ships |
|---|---|
| **09** Just enough Terraform | the smallest IaC vocabulary the port needs |
| **10** Azure platform foundation | the Part I footprint rebuilt from managed services |
| **11** Porting jobs & apps to ACA | every Compose workload mapped to its ACA counterpart, verified against the contract |
| **12** CI/CD: build, scan, deploy | images built and scanned in CI; deploys are digest bumps |
| **13** Operations on Azure | Easy Auth, results/run inspection, two log-backed batch alerts, and cost guardrails |

**Off the critical path:** **14** multi-GPU training is an admission-gated
exception that does not block the golden path in either part.

**Capstone:** chapter **15** integrates the whole golden path end to end through
separate local and Azure adapters. The trigger mechanisms differ, but both suites
preserve the same definition of done: terminal success, evaluation evidence,
results state, exact model identity, readiness, and prediction behavior.


## How to read the rest of the course

Each chapter is structured the same way, so the build stays predictable:

- **Outcome** — what you can do when the chapter is done.
- **Design** — the architecture decisions and tradeoffs introduced by the
  chapter.
- **Build in `projects/ml-platform/`** — the modules, images, IaC, or scripts the
  chapter adds to the project (the source is authored in the project, not the
  notebook).
- **Golden-path position** — where this chapter's slice sits in the diagram above.
- **Acceptance evidence** — what demonstrates the slice actually works, in the
  environment that chapter builds on.
- **Extensions** — optional capabilities or production hardening beyond the
  chapter's baseline.

Next: **[02 — Local platform foundation](./02-local-foundation.ipynb)** stands up
the footprint every later chapter depends on.
